# 02 — The encoding pipeline & fidelity trade-offs

<sup>(C) Copyright 2026- ECMWF and individual contributors. Licensed under the Apache Licence, Version 2.0.</sup>

**What you will learn in this notebook**

- The three stages of the Tensogram pipeline: *encoding → filter → compression*.
- How to pick a codec combination for your data — lossless vs lossy, fast vs small.
- Measure compression ratio **and** round-trip fidelity side-by-side.
- Use `tensogram.compute_packing_params()` to pre-compute GRIB-style `simple_packing` parameters.
- Understand when to use `shuffle` as a compression *amplifier*.

**Prerequisites**: see [`README.md`](README.md). No OS-level C libraries needed — every codec built into the default Python wheel is available.

In [ ]:
import matplotlib

matplotlib.use("Agg")

import time

import matplotlib.pyplot as plt
import numpy as np

import tensogram

# Synthetic "temperature-like" field on a coarse global grid.
# ~700k elements — small enough to run all 8 codecs in a few seconds.
NPOINTS = 720
lats = np.linspace(-90, 90, NPOINTS // 2)
lons = np.linspace(-180, 180, NPOINTS)
LAT, LON = np.meshgrid(lats, lons, indexing="ij")

raw = (
    273.15
    + 30 * np.cos(np.deg2rad(LAT))
    - 10 * np.cos(np.deg2rad(2 * LON))
    + 3.0 * np.random.default_rng(seed=42).standard_normal(LAT.shape)
).astype(np.float64)  # float64 so simple_packing is always applicable

print(f"field shape:  {raw.shape}")
print(f"field size:   {raw.nbytes / 1024:.1f} KiB")
print(f"range:        [{raw.min():.2f}, {raw.max():.2f}] K")
print(f"std:          {raw.std():.4f}")

## 1. The three pipeline stages

Every data object in Tensogram passes through three fixed stages on encode, applied in order:

```
raw bytes → encoding → filter → compression → payload on wire
```

| Stage | Options | Purpose |
|---|---|---|
| **encoding** | `none`, `simple_packing` | Transform the representation — e.g. quantize floats into packed integers. |
| **filter** | `none`, `shuffle` | Reorder bytes so compressors see smoother patterns. |
| **compression** | `none`, `lz4`, `zstd`, `szip`, `blosc2`, `zfp`, `sz3` | Shrink the bytes. Some are lossless; some are lossy. |

Inverse stages run automatically on decode.

You choose by adding keys to the descriptor. Here's the whole API surface:

In [ ]:
def encode_with_pipeline(raw_array, *, encoding="none", bits=None,
                        filter="none", compression="none",
                        compression_level=None, metadata_extra=None):
    """Encode a float64 field with the given pipeline and return
    (message_bytes, encode_time, decoded_array, decode_time)."""
    descriptor = {
        "type": "ntensor",
        "shape": list(raw_array.shape),
        "dtype": str(raw_array.dtype),
        "encoding": encoding,
        "filter": filter,
        "compression": compression,
    }
    # Pipeline-specific parameters. The full set is documented in
    # docs/src/encodings/compression.md — we use the most common ones here.
    if encoding == "simple_packing":
        assert bits is not None, "simple_packing needs a bits_per_value"
        params = tensogram.compute_packing_params(
            raw_array.ravel().astype(np.float64),
            bits_per_value=bits,
            decimal_scale_factor=0,
        )
        descriptor.update(params)  # reference_value, binary_scale_factor, ...
    if filter == "shuffle":
        descriptor["shuffle_element_size"] = raw_array.dtype.itemsize
    if compression == "zstd" and compression_level is not None:
        descriptor["zstd_level"] = compression_level
    if compression == "szip":
        # szip requires three mandatory params. These defaults match what
        # `tensogram convert-grib` picks via the shared pipeline helper.
        descriptor.setdefault("szip_rsi", 128)
        descriptor.setdefault("szip_block_size", 16)
        descriptor.setdefault("szip_flags", 8)

    meta = {}
    if metadata_extra:
        meta.update(metadata_extra)

    t0 = time.perf_counter()
    msg = bytes(tensogram.encode(meta, [(descriptor, raw_array)]))
    enc_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    _meta, objs = tensogram.decode(msg)
    dec_ms = (time.perf_counter() - t0) * 1000

    return msg, enc_ms, objs[0][1], dec_ms


# Quick smoke test — no compression, no encoding.
msg, enc_ms, decoded, dec_ms = encode_with_pipeline(raw)
print(f"raw pass-through: {len(msg):,} bytes  enc={enc_ms:.1f}ms  dec={dec_ms:.1f}ms")
assert np.array_equal(decoded, raw), "pass-through must be byte-identical"
print("round-trip OK")

## 2. Sweeping codec combinations

Let's measure how 8 codec combinations perform on our synthetic temperature field. We compute:

- **ratio** = compressed size / raw size (lower is better)
- **encode time** (ms)
- **decode time** (ms)
- **Linf error** = max absolute difference between decoded and original (0 = lossless)

The `none`/`none`/`none` row is the reference.

In [ ]:
CASES = [
    # (name, encoding, bits, filter, compression, compression_level)
    ("none",                    "none",            None, "none",    "none",   None),
    ("lz4",                     "none",            None, "none",    "lz4",    None),
    ("zstd-3",                  "none",            None, "none",    "zstd",   3),
    ("zstd-12",                 "none",            None, "none",    "zstd",   12),
    ("shuffle + zstd-3",        "none",            None, "shuffle", "zstd",   3),
    ("simple_packing 16-bit",   "simple_packing",  16,   "none",    "none",   None),
    ("simple_packing 16 + szip", "simple_packing", 16,   "none",    "szip",   None),
    ("simple_packing 24 + zstd-3", "simple_packing", 24,  "none",    "zstd",   3),
]

raw_bytes = raw.nbytes
rows = []
for name, enc, bits, filt, comp, lvl in CASES:
    msg, enc_ms, decoded, dec_ms = encode_with_pipeline(
        raw, encoding=enc, bits=bits, filter=filt,
        compression=comp, compression_level=lvl,
    )
    linf = float(np.max(np.abs(decoded.astype(np.float64) - raw)))
    rows.append({
        "name": name, "size": len(msg),
        "ratio": len(msg) / raw_bytes,
        "enc_ms": enc_ms, "dec_ms": dec_ms, "linf": linf,
    })
    print(f"{name:<32}  {len(msg):>9,} B  {len(msg)/raw_bytes*100:5.1f}%  "
          f"enc={enc_ms:5.1f}ms  dec={dec_ms:5.1f}ms  Linf={linf:.3e}")

### Plot: size vs encode time

The right-most point in the plot below is the uncompressed baseline. The **bottom-left corner** is the best trade-off (fast **and** small). Lossy codecs (Linf > 0) typically move furthest down-left.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# (a) compression ratio bar chart
names = [r["name"] for r in rows]
ratios = [r["ratio"] * 100 for r in rows]
colors = ["#aaaaaa" if r["linf"] == 0 else "#e76f51" for r in rows]
axes[0].barh(names, ratios, color=colors)
axes[0].axvline(100, color="black", linestyle="--", alpha=0.5, label="raw size")
axes[0].set_xlabel("compressed size (% of raw)")
axes[0].set_title("Compression ratio\n(grey = lossless, orange = lossy)")
axes[0].invert_yaxis()
axes[0].grid(axis="x", alpha=0.3)

# (b) scatter: encode time vs size
axes[1].scatter(
    [r["enc_ms"] for r in rows],
    [r["ratio"] * 100 for r in rows],
    c=colors, s=80, edgecolor="black",
)
for r in rows:
    axes[1].annotate(r["name"], (r["enc_ms"], r["ratio"] * 100),
                     textcoords="offset points", xytext=(5, 5), fontsize=8)
axes[1].set_xlabel("encode time (ms)")
axes[1].set_ylabel("size (% of raw)")
axes[1].set_title("Time vs size trade-off")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Fidelity trade-off for `simple_packing`

`simple_packing` is **lossy** — it quantizes floats into N-bit integers before compression. The quantization step is proportional to `2^(-N) × (max - min)`. At 16 bits the error is ~1e-4 K for a typical temperature field; at 8 bits it becomes visible on a map.

In [ ]:
fidelity_rows = []
for bits in (8, 12, 16, 20, 24, 32):
    msg, _enc_ms, decoded, _dec_ms = encode_with_pipeline(
        raw, encoding="simple_packing", bits=bits, compression="zstd",
        compression_level=3,
    )
    linf = float(np.max(np.abs(decoded - raw)))
    rms = float(np.sqrt(np.mean((decoded - raw) ** 2)))
    fidelity_rows.append((bits, len(msg), linf, rms))
    print(f"bits={bits:2d}  size={len(msg):>8,} B  "
          f"({len(msg)/raw.nbytes*100:5.2f}% of raw)  Linf={linf:.2e}  RMS={rms:.2e}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
bits_arr = [r[0] for r in fidelity_rows]
ax1.plot(bits_arr, [r[1] / 1024 for r in fidelity_rows], "o-")
ax1.set_xlabel("bits per value")
ax1.set_ylabel("message size (KiB)")
ax1.set_title("Size vs quantization precision")
ax1.grid(alpha=0.3)

ax2.semilogy(bits_arr, [r[2] for r in fidelity_rows], "o-", label="Linf")
ax2.semilogy(bits_arr, [r[3] for r in fidelity_rows], "s--", label="RMS")
ax2.set_xlabel("bits per value")
ax2.set_ylabel("error (K)")
ax2.set_title("Fidelity vs quantization precision")
ax2.grid(alpha=0.3, which="both")
ax2.legend()

plt.tight_layout()
plt.show()

### Visual check: where does 8-bit quantization hurt?

Difference map between the original and the 8-bit `simple_packing` round-trip. For a continuous field like temperature, 8-bit is too aggressive; 16-bit or above is fine for most scientific workflows.

In [ ]:
_, _, decoded_8bit, _ = encode_with_pipeline(
    raw, encoding="simple_packing", bits=8, compression="zstd",
)
diff = decoded_8bit - raw

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
im1 = a1.imshow(raw, origin="lower", cmap="RdYlBu_r", aspect="auto")
a1.set_title("Original field (K)")
plt.colorbar(im1, ax=a1)

im2 = a2.imshow(diff, origin="lower", cmap="RdBu", aspect="auto")
a2.set_title(f"8-bit simple_packing error\nLinf={np.abs(diff).max():.3f} K")
plt.colorbar(im2, ax=a2)
plt.tight_layout()
plt.show()

## 4. The `shuffle` filter — a free compression amplifier

`shuffle` byte-transposes a typed array so that all first bytes of each element come first, then all second bytes, etc. The compressor downstream sees long runs of similar bytes and does better.

Shuffle is **lossless** and cheap. It's almost always worth turning on with `zstd` or `lz4` on typed numeric data.

In [ ]:
for comp, lvl in [("zstd", 3), ("zstd", 12), ("lz4", None)]:
    msg_plain, *_ = encode_with_pipeline(raw, compression=comp, compression_level=lvl)
    msg_shuf, *_ = encode_with_pipeline(
        raw, filter="shuffle", compression=comp, compression_level=lvl,
    )
    improvement = (1 - len(msg_shuf) / len(msg_plain)) * 100
    print(f"{comp}{'-' + str(lvl) if lvl else '':<4}  "
          f"plain={len(msg_plain):>8,}  shuffled={len(msg_shuf):>8,}  "
          f"shuffle saves {improvement:5.1f}%")

## 5. Lossy codecs: `zfp` and `sz3`

For floating-point arrays where you accept a bounded error, `zfp` and `sz3` are the specialists. Both have multiple error modes — absolute, relative, PSNR.

In [ ]:
# zfp fixed-accuracy with tolerance 0.01 K — ideal for fields where
# we care about max-error, not average error.
descriptor = {
    "type": "ntensor", "shape": list(raw.shape), "dtype": "float64",
    "encoding": "none", "filter": "none",
    "compression": "zfp",
    "zfp_mode": "fixed_accuracy",
    "zfp_tolerance": 0.01,
}
msg_zfp = bytes(tensogram.encode({}, [(descriptor, raw)]))
_, objs = tensogram.decode(msg_zfp)
decoded_zfp = objs[0][1]
print(f"zfp fixed_accuracy=0.01:  {len(msg_zfp):,} B  "
      f"({len(msg_zfp)/raw.nbytes*100:5.2f}% of raw)  "
      f"Linf={np.abs(decoded_zfp - raw).max():.4f} K")

# sz3 absolute error bound — similar idea, different algorithm.
descriptor = {
    "type": "ntensor", "shape": list(raw.shape), "dtype": "float64",
    "encoding": "none", "filter": "none",
    "compression": "sz3",
    "sz3_error_bound_mode": "abs",
    "sz3_error_bound": 0.01,
}
msg_sz3 = bytes(tensogram.encode({}, [(descriptor, raw)]))
_, objs = tensogram.decode(msg_sz3)
decoded_sz3 = objs[0][1]
print(f"sz3  abs_error=0.01:      {len(msg_sz3):,} B  "
      f"({len(msg_sz3)/raw.nbytes*100:5.2f}% of raw)  "
      f"Linf={np.abs(decoded_sz3 - raw).max():.4f} K")

## 6. Practical decision guide

- **Need exact values, don't care about speed** → `simple_packing` off, `shuffle` on, `zstd` level 12.
- **Need exact values, need speed** → `lz4` with `shuffle`.
- **Weather/climate fields, GRIB interop** → `simple_packing` 16–24 bits + `szip`. Byte-identical to ecCodes.
- **Any float field, bounded error acceptable** → `zfp` fixed_accuracy, or `sz3` abs.
- **Large sparse/uniform arrays** → `blosc2` (internal LZ4 + shuffle, multi-threaded).

Full reference: [`docs/src/encodings/compression.md`](../../docs/src/encodings/compression.md).

## Where to go next

- [`03_from_grib_to_tensogram.ipynb`](03_from_grib_to_tensogram.ipynb) — apply this pipeline to real ECMWF opendata.
- [`05_validation_and_parallelism.ipynb`](05_validation_and_parallelism.ipynb) — scale encoding across threads.